# Webサイトの要約

In [1]:
import os
import random
import re
import pandas as pd
from bs4 import BeautifulSoup
from charset_normalizer import from_path
# from LangExtract import summarize # Assuming LangExtract is an available library

# Placeholder for LangExtract summarize function
def summarize(text):
    # This is a placeholder. In a real scenario, this would call the LangExtract library.
    # For now, it just returns the first 200 characters as a mock summary.
    return text[:200] + '...'

/var/folders/w2/q44s18pj53xcm4z7b2kmstxw0000gp/T/ipykernel_17506/2859406317.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## HTMLからのテキスト抽出

`dorasite_analysis.ipynb` を参考に、HTMLファイルからテキストを抽出する関数を定義します。

In [2]:
def is_mojibake(text):
    """
    簡易的な文字化け検出関数。非表示文字や無効な文字がある場合にTrueを返す。
    """
    mojibake_patterns = [
        r'[]'  # 置換文字
    ]
    
    for pattern in mojibake_patterns:
        if re.search(pattern, text):
            return True
    return False

def extract_text_from_html(file_path):
    """
    HTMLファイルからテキストを抽出する関数
    """
    try:
        result = from_path(file_path).best()
        content = str(result)
        soup = BeautifulSoup(content, 'html.parser')
        text = soup.get_text(separator=' ')
        cleaned_text = re.sub(r'\s+', ' ', text).strip()

        if is_mojibake(cleaned_text):
            # print(f"Mojibake detected in {file_path}, skipping.")
            return None
        
        return cleaned_text
    except Exception as e:
        # print(f"Error reading {file_path}: {e}")
        return None

## ファイルの探索とランダムサンプリング

対象となるディレクトリを再帰的に探索し、HTMLファイルのリストを作成します。その後、リストからランダムに100ページを抽出します。

In [3]:
def find_html_files(directories):
    html_files = []
    for directory in directories:
        for root, _, files in os.walk(directory):
            for file in files:
                if file.endswith(('.html', '.htm')):
                    html_files.append(os.path.join(root, file))
    return html_files

# 対象ディレクトリのリスト
target_dirs = [
    '2nd.geocities.jp',
    'anime.geocities.jp',
    'hen-dora.com',
    'sky.geocities.jp',
    'www.geocities.co.jp',
    'www.geocities.jp'
]

all_files = find_html_files(target_dirs)
print(f"Found {len(all_files)} HTML files in total.")

# 100ページをランダムにサンプリング
if len(all_files) > 100:
    sampled_files = random.sample(all_files, 100)
else:
    sampled_files = all_files

print(f"Selected {len(sampled_files)} files for summarization.")

Found 4009 HTML files in total.
Selected 100 files for summarization.


## 要約の実行と結果の表示

サンプリングされた各ファイルについて、テキストを抽出して要約を実行し、結果をDataFrameに格納します。

In [5]:
summaries = []

for file_path in sampled_files:
    text = extract_text_from_html(file_path)
    if text:
        summary = summarize(text)
        summaries.append({
            'file_path': file_path,
            'summary': summary
        })

summary_df = pd.DataFrame(summaries)

summary_df.head()

""
